In [0]:
devices_df = spark.read.csv("/Volumes/hexa_databricks_7405605275266839/default/datasets/devices.csv", header=True, inferSchema=True)
energy_df = spark.read.csv("/Volumes/hexa_databricks_7405605275266839/default/datasets/energy_logs.csv", header=True, inferSchema=True)

In [0]:
devices_df.printSchema()
energy_df.printSchema()

root
 |-- device_id: integer (nullable = true)
 |-- device_name: string (nullable = true)
 |-- room_id: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- power_rating_watts: integer (nullable = true)

root
 |-- device_id: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- usage_hours: integer (nullable = true)
 |-- time_of_day: string (nullable = true)



In [0]:
display(devices_df)

device_id,device_name,room_id,device_type,power_rating_watts
101,Fridge,R1,Appliance,200
102,AC,R4,Cooling,1500
103,Washing Machine,R2,Appliance,500
104,TV,R3,Electronics,150
105,Heater,R2,Heating,2000
106,Microwave,R4,Appliance,1200


In [0]:
display(energy_df)

device_id,date,energy_kwh,usage_hours,time_of_day
105,2025-01-02,9.6,12,Peak
104,2025-01-02,9.6,3,Peak
103,2025-01-03,9.2,8,Peak
102,2025-01-02,7.4,5,Off-Peak
106,2025-01-03,1.5,10,Peak
103,2025-01-01,1.1,5,Peak
103,2025-01-04,2.1,7,Off-Peak
101,2025-01-01,4.5,11,Peak
106,2025-01-01,2.3,4,Off-Peak
102,2025-01-03,3.2,7,Off-Peak


In [0]:
combined_df = energy_df.join(
    devices_df, 
    "device_id",
    "left")
combined_df.show()

+---------+----------+----------+-----------+-----------+---------------+-------+-----------+------------------+
|device_id|      date|energy_kwh|usage_hours|time_of_day|    device_name|room_id|device_type|power_rating_watts|
+---------+----------+----------+-----------+-----------+---------------+-------+-----------+------------------+
|      105|2025-01-02|       9.6|         12|       Peak|         Heater|     R2|    Heating|              2000|
|      104|2025-01-02|       9.6|          3|       Peak|             TV|     R3|Electronics|               150|
|      103|2025-01-03|       9.2|          8|       Peak|Washing Machine|     R2|  Appliance|               500|
|      102|2025-01-02|       7.4|          5|   Off-Peak|             AC|     R4|    Cooling|              1500|
|      106|2025-01-03|       1.5|         10|       Peak|      Microwave|     R4|  Appliance|              1200|
|      103|2025-01-01|       1.1|          5|       Peak|Washing Machine|     R2|  Appliance|   

In [0]:
from pyspark.sql.functions import *

room_summary_df = combined_df.groupBy("room_id", "date").agg(
    sum("energy_kwh").alias("total_energy_kwh")
)
room_summary_df.show()

+-------+----------+-----------------+
|room_id|      date| total_energy_kwh|
+-------+----------+-----------------+
|     R2|2025-01-02|              9.6|
|     R2|2025-01-03|              9.2|
|     R4|2025-01-03|              4.7|
|     R2|2025-01-04|              2.1|
|     R2|2025-01-01|              1.1|
|     R1|2025-01-02|              7.8|
|     R4|2025-01-01|9.399999999999999|
|     R4|2025-01-05|              3.3|
|     R1|2025-01-01|              4.5|
|     R3|2025-01-04|              0.6|
|     R2|2025-01-05|              5.1|
|     R3|2025-01-02|              9.6|
|     R4|2025-01-02|              7.4|
|     R3|2025-01-03|              2.2|
|     R1|2025-01-03|              3.7|
+-------+----------+-----------------+



In [0]:
device_summary_df = combined_df.groupBy("device_id", "device_name", "device_type", "power_rating_watts").agg(
    sum("energy_kwh").alias("total_energy_used")
)
device_summary_df.show()

+---------+---------------+-----------+------------------+------------------+
|device_id|    device_name|device_type|power_rating_watts| total_energy_used|
+---------+---------------+-----------+------------------+------------------+
|      105|         Heater|    Heating|              2000|              14.7|
|      102|             AC|    Cooling|              1500|19.800000000000004|
|      101|         Fridge|  Appliance|               200|              16.0|
|      103|Washing Machine|  Appliance|               500|12.399999999999999|
|      104|             TV|Electronics|               150|              12.4|
|      106|      Microwave|  Appliance|              1200|               5.0|
+---------+---------------+-----------+------------------+------------------+



In [0]:
peak_summary_df = combined_df.groupBy("device_id", "device_name", "time_of_day").agg(
    sum("energy_kwh").alias("total_kwh")
)
peak_summary_df.show()

+---------+---------------+-----------+------------------+
|device_id|    device_name|time_of_day|         total_kwh|
+---------+---------------+-----------+------------------+
|      106|      Microwave|       Peak|               1.5|
|      103|Washing Machine|       Peak|10.299999999999999|
|      101|         Fridge|       Peak|               4.5|
|      102|             AC|       Peak|               7.1|
|      104|             TV|       Peak|              12.4|
|      101|         Fridge|   Off-Peak|              11.5|
|      105|         Heater|       Peak|              14.7|
|      106|      Microwave|   Off-Peak|               3.5|
|      102|             AC|   Off-Peak|12.700000000000001|
|      103|Washing Machine|   Off-Peak|               2.1|
+---------+---------------+-----------+------------------+



In [0]:
alert_df = combined_df.withColumn(
    "alert",
    when(col("energy_kwh") > 8, "High Usage").otherwise("Normal")
)
alert_df.show()

+---------+----------+----------+-----------+-----------+---------------+-------+-----------+------------------+----------+
|device_id|      date|energy_kwh|usage_hours|time_of_day|    device_name|room_id|device_type|power_rating_watts|     alert|
+---------+----------+----------+-----------+-----------+---------------+-------+-----------+------------------+----------+
|      105|2025-01-02|       9.6|         12|       Peak|         Heater|     R2|    Heating|              2000|High Usage|
|      104|2025-01-02|       9.6|          3|       Peak|             TV|     R3|Electronics|               150|High Usage|
|      103|2025-01-03|       9.2|          8|       Peak|Washing Machine|     R2|  Appliance|               500|High Usage|
|      102|2025-01-02|       7.4|          5|   Off-Peak|             AC|     R4|    Cooling|              1500|    Normal|
|      106|2025-01-03|       1.5|         10|       Peak|      Microwave|     R4|  Appliance|              1200|    Normal|
|      1

In [0]:
room_summary_df.write.format("delta").mode("overwrite").save("/Volumes/hexa_databricks_7405605275266839/default/datasets/room_energy_summary_delta")

alert_df.write.format("csv").mode("overwrite").option("header", True).save("/Volumes/hexa_databricks_7405605275266839/default/datasets/device_alerts_csv")

print("ETL pipeline finished! Files saved.")

ETL pipeline finished! Files saved.


In [0]:
join_df = energy_df.join(devices_df, on="device_id", how="left")

devices_tot = join_df.groupBy("device_id", "device_name").agg(
    sum("energy_kwh").alias("total_energy")
)

print("Report: ")
print("\nTotal devices: ", devices_df.count())
print("Total energy logs: ", energy_df.count())

print("\nDevice with most energy usage: ")
devices_tot.orderBy(col("total_energy").desc()).show(1)

print("\nDevice with least energy usage: ")
devices_tot.orderBy(col("total_energy").asc()).show(1)

print("\nHighest single energy reading: ")
join_df.orderBy(col("energy_kwh").desc()).show(1)

print("\nLowest single energy reading: ")
join_df.orderBy(col("energy_kwh").asc()).show(1)

print("\nTotal energy usage by room: ")
join_df.groupBy("room_id").sum("energy_kwh").show()

print("\nTotal energy usage by device: ")
devices_tot.show()

Report: 

Total devices:  6
Total energy logs:  18

Device with most energy usage: 
+---------+-----------+------------------+
|device_id|device_name|      total_energy|
+---------+-----------+------------------+
|      102|         AC|19.800000000000004|
+---------+-----------+------------------+
only showing top 1 row

Device with least energy usage: 
+---------+-----------+------------+
|device_id|device_name|total_energy|
+---------+-----------+------------+
|      106|  Microwave|         5.0|
+---------+-----------+------------+
only showing top 1 row

Highest single energy reading: 
+---------+----------+----------+-----------+-----------+-----------+-------+-----------+------------------+
|device_id|      date|energy_kwh|usage_hours|time_of_day|device_name|room_id|device_type|power_rating_watts|
+---------+----------+----------+-----------+-----------+-----------+-------+-----------+------------------+
|      105|2025-01-02|       9.6|         12|       Peak|     Heater|     R2